# 🏠 House Recognition — ConvNeXt Base v5
แก้ครบทุก edge case:
- **Deterministic TTA chunking** (ไม่มี try/except OOM anti-pattern)
- **Gradient Checkpointing** (~40-50% VRAM ลดลง → batch 16 ได้)
- `timm.loss.SoftTargetCrossEntropy` แทน custom mixup
- ลบ vertical flip ออก (บ้านมี gravity)
- Modular cells debug ได้ง่าย


In [1]:
!pip -q install kaggle timm==1.0.15 albumentations==1.4.18 \\
    opencv-python-headless==4.10.0.84 scikit-learn==1.6.1 \\
    pandas==2.2.2 pillow==11.3.0 psutil


ERROR: Invalid requirement: '': Expected package name at the start of dependency specifier
    
    ^


## Cell 2 — ตั้งค่า Kaggle API

In [2]:
from google.colab import files
files.upload()  # อัปโหลด kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


## Cell 3 — ดาวน์โหลดข้อมูล

In [3]:
COMPETITION = "super-ai-engineer-season-6-individual-hackathon-house-recognition"
DATA_ROOT   = "/content/data"

import os
os.makedirs(DATA_ROOT, exist_ok=True)
!kaggle competitions download -c {COMPETITION} -p {DATA_ROOT}
!unzip -qo {DATA_ROOT}/{COMPETITION}.zip -d {DATA_ROOT}


100% 1.25G/1.25G [00:31<00:00, 42.3MB/s]



## Cell 4 — ตรวจสอบข้อมูล

In [4]:
import os, pandas as pd

DATA_DIR    = "/content/data"
TRAIN_DIR   = f"{DATA_DIR}/train/train"
TEST_DIR    = f"{DATA_DIR}/test/test"

df = pd.read_csv(f"{DATA_DIR}/train.csv")
print(f"Train rows : {len(df)}")
print(f"Class dist : {df['class'].value_counts().to_dict()}")
print(f"Train imgs : {len(os.listdir(TRAIN_DIR))}")
print(f"Test  imgs : {len(os.listdir(TEST_DIR))}")
df.head()


Train rows : 2953
Class dist : {0: 1520, 1: 1433}
Train imgs : 2954
Test  imgs : 1550


,image_name,class
0,ChokChai4_img_13-7956791_100-6031267_a187-2159...,0
1,ChokChai4_img_13-7961753_100-6031881_a185-9785...,0
2,ChokChai4_img_13-7969811_100-5906061_a180-5812...,0
3,ChokChai4_img_13-7970811_100-5906071_a180-5812...,0
4,ChokChai4_img_13-7971811_100-5906081_a180-5812...,0


## Cell 5 — Imports & Config
> แก้ที่นี่เพื่อเปลี่ยน hyperparameter ไม่ต้องแก้ใน script


In [5]:
from __future__ import annotations
import gc, math, os, random
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import albumentations as A
import cv2, numpy as np, pandas as pd
import timm
import torch, torch.nn as nn, torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from timm.loss import SoftTargetCrossEntropy  # Fix #2
from tqdm.auto import tqdm

@dataclass
class CFG:
    MODEL_NAME: str     = "convnext_base.fb_in22k_ft_in1k_384"
    SEED: int           = 42
    N_SPLITS: int        = 5
    IMAGE_SIZE: int      = 384
    TRAIN_BS: int        = 16   # เพิ่มได้เพราะ Grad Checkpointing ลด VRAM ~40-50%
    VALID_BS: int        = 32
    GRAD_ACCUM: int      = 1    # ไม่ต้อง accum แล้วเพราะ batch ใหญ่พอ
    EPOCHS: int          = 12
    LR: float            = 3e-5
    MIN_LR: float        = 1e-7
    WEIGHT_DECAY: float  = 1e-2
    WARMUP_EPOCHS: int   = 2
    NUM_WORKERS: int     = 2
    PREFETCH: int        = 4
    LABEL_SMOOTH: float  = 0.1
    MIXUP_ALPHA: float   = 0.4
    TTA: bool            = True
    TTA_VIEWS: int       = 2    # 2=original+hflip (ไม่มี vflip เพราะบ้านมี gravity)
    TTA_CHUNKS: int      = 2    # แบ่ง mega-batch เป็น N chunks ป้องกัน OOM
    AMP: bool            = True
    GRAD_CLIP: float     = 1.0
    LAYER_DECAY: float   = 0.85
    PATIENCE: int        = 4
    CACHE_IMAGES: bool   = True
    CHANNELS_LAST: bool  = True
    GRAD_CKPT: bool      = True  # -40-50% VRAM, +20% compute
    NUM_CLASSES: int     = 2
    DATA_DIR: str       = "/content/data"
    OUTPUT_DIR: str     = "/content/outputs"

cfg = CFG()
print(f"Model          : {cfg.MODEL_NAME}")
print(f"Image size     : {cfg.IMAGE_SIZE}")
print(f"Train batch    : {cfg.TRAIN_BS} (GRAD_CKPT={cfg.GRAD_CKPT})")


Model          : convnext_base.fb_in22k_ft_in1k_384
Image size     : 384
Train batch    : 16 (GRAD_CKPT=True)


## Cell 6 — Seeding & Image Cache

In [6]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def worker_init_fn(worker_id: int) -> None:
    """Fix: ป้องกัน fork() copy numpy seed เหมือนกันทุก worker"""
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def build_image_cache(image_dir: Path, image_names: list) -> dict | None:
    """ตรวจ RAM ก่อน cache — ป้องกัน silent OOM kill"""
    import psutil
    estimated  = len(image_names) * cfg.IMAGE_SIZE * cfg.IMAGE_SIZE * 3
    available  = psutil.virtual_memory().available
    margin     = 2 * 1024**3  # เก็บไว้ 2GB
    if estimated > available - margin:
        print(f"[Cache] ข้ามการ cache: ต้องการ {estimated/1e9:.1f}GB > available {available/1e9:.1f}GB")
        return None
    cache = {}
    print(f"[Cache] โหลด {len(image_names)} ภาพ...")
    for name in tqdm(image_names, desc='cache', leave=False):
        img = cv2.imread(str(image_dir / name))
        if img is None: raise FileNotFoundError(f'Cannot read: {image_dir/name}')
        cache[name] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"[Cache] {sum(v.nbytes for v in cache.values())/1e9:.2f} GB used")
    return cache

seed_everything(cfg.SEED)
print('Seed set:', cfg.SEED)


Seed set: 42


## Cell 7 — Transforms
> **Fix #3 — Data Invariance:** ลบ vertical flip ออก
> บ้านมี gravity — ภาพหัวกลับไม่มีในธรรมชาติ ใส่ไว้ทำลาย feature เปล่าๆ
> TTA ก็ลด view จาก 4 เหลือ 2 (original + hflip เท่านั้น)


In [7]:
# ── albumentations >=1.4 ใหม่ ──

def build_train_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=img_size, interpolation=cv2.INTER_LINEAR),
        A.PadIfNeeded(
            min_height=img_size, min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=114,                           # ✅ เปลี่ยนจาก value= → fill=
        ),
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.75, 1.0), ratio=(0.85, 1.15),
            interpolation=cv2.INTER_LINEAR,
        ),
        A.HorizontalFlip(p=0.5),
        A.Affine(                               # ✅ เปลี่ยนจาก ShiftScaleRotate → Affine
            translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            border_mode=cv2.BORDER_CONSTANT,
            fill=114,
            p=0.3,
        ),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.1)),  # ✅ เปลี่ยนจาก var_limit= → std_range=
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=7),
        ], p=0.25),
        A.OneOf([
            A.RandomRain(blur_value=2, p=1.0),
            A.RandomFog(fog_coef_range=(0.1, 0.3), p=1.0),
            A.RandomShadow(p=1.0),
        ], p=0.15),
        A.CoarseDropout(                          # ✅ API ใหม่ทั้งหมด
            num_holes_range=(1, 6),
            hole_height_range=(int(img_size*0.04), int(img_size*0.18)),
            hole_width_range=(int(img_size*0.04), int(img_size*0.18)),
            fill=114,
            p=0.3,
        ),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


def build_valid_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=img_size, interpolation=cv2.INTER_LINEAR),
        A.PadIfNeeded(
            min_height=img_size, min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=114,                           # ✅
        ),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

print("Transforms ready — no warnings")

Transforms ready — no warnings


## Cell 8 — Dataset & Model

In [8]:
class HouseDataset(Dataset):
    def __init__(self, df, image_dir, transforms, is_test=False, cache=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transforms = transforms
        self.is_test = is_test
        self.cache = cache

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        name = row['image_name']
        if self.cache is not None:
            image = self.cache[name].copy()
        else:
            image = cv2.imread(str(self.image_dir / name))
            if image is None: raise FileNotFoundError(f'Cannot read: {name}')
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)['image']
        if self.is_test: return image, row['id']
        return image, torch.tensor(int(row['class']), dtype=torch.long)


class HouseModel(nn.Module):
    def __init__(self, model_name, num_classes=2, grad_ckpt=False):
        super().__init__()
        self.model = timm.create_model(
            model_name, pretrained=True, num_classes=num_classes,
            drop_rate=0.2, drop_path_rate=0.2,
        )
        # Fix #3 — Gradient Checkpointing
        # trade ~20% compute for ~40-50% VRAM reduction
        # allows batch 16 on T4 without OOM
        if grad_ckpt:
            self.model.set_grad_checkpointing(enable=True)
    def forward(self, x): return self.model(x)

print('Dataset & Model defined')


Dataset & Model defined


## Cell 9 — Optimizer, Scheduler, Mixup
> **Fix #2:** ใช้ `timm.loss.SoftTargetCrossEntropy` แทน custom function


In [9]:
def build_layerwise_optimizer(model, base_lr, weight_decay, layer_decay):
    head_params = list(model.model.head.parameters())
    head_ids    = {id(p) for p in head_params}
    groups = [{'params': head_params, 'lr': base_lr, 'weight_decay': weight_decay}]
    for depth, name in enumerate(['stages.3','stages.2','stages.1','stages.0','stem'], 1):
        lr = base_lr * (layer_decay ** depth)
        params = [p for n,p in model.named_parameters() if name in n and id(p) not in head_ids]
        if params: groups.append({'params': params, 'lr': lr, 'weight_decay': weight_decay})
    return AdamW(groups)

def build_scheduler(optimizer, total_steps, warmup_steps, min_lr, base_lr):
    def lr_lambda(step):
        if step < warmup_steps: return float(step+1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return (min_lr/base_lr) + (1.0 - min_lr/base_lr) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_data(x, y_onehot, alpha=0.4):
    """Fix #2: ทำ one-hot ก่อน mixup เพื่อส่งให้ SoftTargetCrossEntropy"""
    lam   = float(np.random.beta(alpha, alpha))
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x      = lam * x + (1-lam) * x[index]
    mixed_target = lam * y_onehot + (1-lam) * y_onehot[index]
    return mixed_x, mixed_target, lam

# SoftTargetCrossEntropy จาก timm — handle label smoothing + soft target ถูกต้อง
soft_criterion = SoftTargetCrossEntropy()
hard_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTH)

print('Optimizer, Scheduler, Loss ready')


Optimizer, Scheduler, Loss ready


## Cell 10 — Train & Validate Functions

In [10]:
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler):
    model.train()
    losses, accs = [], []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc='train', leave=False)
    for step, (images, labels) in enumerate(pbar):
        if cfg.CHANNELS_LAST:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        else:
            images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        use_mixup = cfg.MIXUP_ALPHA > 0 and random.random() < 0.5
        if use_mixup:
            # สร้าง smoothed one-hot แล้วส่งให้ SoftTargetCrossEntropy
            smooth   = cfg.LABEL_SMOOTH / cfg.NUM_CLASSES
            y_onehot = F.one_hot(labels, cfg.NUM_CLASSES).float()
            y_onehot = y_onehot * (1 - cfg.LABEL_SMOOTH) + smooth
            images, mixed_target, _ = mixup_data(images, y_onehot, cfg.MIXUP_ALPHA)

        with torch.amp.autocast(device_type=device.type, enabled=cfg.AMP):
            logits = model(images)
            loss   = soft_criterion(logits, mixed_target) if use_mixup else hard_criterion(logits, labels)
            loss   = loss / cfg.GRAD_ACCUM

        scaler.scale(loss).backward()
        if (step+1) % cfg.GRAD_ACCUM == 0 or (step+1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        acc = (logits.detach().argmax(1) == labels).float().mean().item()
        losses.append(loss.item() * cfg.GRAD_ACCUM)
        accs.append(acc)
        pbar.set_postfix(loss=f'{np.mean(losses):.4f}', acc=f'{np.mean(accs):.4f}')
    return float(np.mean(losses)), float(np.mean(accs))


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    losses, all_probs, all_labels = [], [], []
    for images, labels in tqdm(loader, desc='valid', leave=False):
        if cfg.CHANNELS_LAST:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        else:
            images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        losses.append(hard_criterion(logits, labels).item())
        all_probs.append(torch.softmax(logits, dim=1)[:,1].cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return float(np.mean(losses)), accuracy_score(labels, (probs>=0.5).astype(int)), probs

print('Train/Validate functions ready')


Train/Validate functions ready


## Cell 11 — TTA Predict
> **Fix #1 — Deterministic TTA Chunking (ไม่มี try/except OOM anti-pattern)**
> แบ่ง mega-batch เป็น `TTA_CHUNKS` ชิ้น deterministic → process ทีละชิ้น → concat
> ปลอดภัย 100% ไม่มีโอกาส OOM และ CUDA memory ไม่ fragmented


In [11]:
@torch.no_grad()
def tta_forward(model, images, device):
    """
    Deterministic TTA — ไม่มี try/except OOM anti-pattern
    ขั้นตอน: build views → cat → chunk N → forward ทีละ chunk → cat → split per view → avg
    cfg.TTA_CHUNKS=2: mega-batch 32x2=64 ถูก split เป็น 2x32 → ปลอดภัยบน T4
    """
    views = [images]
    if cfg.TTA_VIEWS >= 2:
        views.append(torch.flip(images, dims=[3]))   # hflip
    # vflip/hvflip ถูกตัดออกแล้ว (บ้านมี gravity)

    if cfg.CHANNELS_LAST:
        views = [v.contiguous(memory_format=torch.channels_last) for v in views]

    # Step 1: concat ทุก view เป็น mega-batch [B*V, C, H, W]
    mega = torch.cat(views, dim=0)

    # Step 2: แบ่งเป็น TTA_CHUNKS chunks อย่าง deterministic
    # ไม่ต้อง try/except → ไม่มี CUDA memory fragmentation
    chunk_outs = []
    for chunk in mega.chunk(cfg.TTA_CHUNKS, dim=0):
        chunk_outs.append(model(chunk))

    # Step 3: concat กลับ แล้ว split ตาม view
    all_logits = torch.cat(chunk_outs, dim=0)
    view_logits = all_logits.chunk(len(views), dim=0)

    # Step 4: average probability ทุก view
    probs = torch.stack(
        [torch.softmax(v, dim=1)[:,1] for v in view_logits], dim=0
    ).mean(0)
    return probs


@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    id_to_prob = {}
    for images, ids in tqdm(loader, desc='predict', leave=False):
        if cfg.CHANNELS_LAST:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        else:
            images = images.to(device, non_blocking=True)
        probs = tta_forward(model, images, device) if cfg.TTA else \
                torch.softmax(model(images), dim=1)[:,1]
        for img_id, prob in zip(ids, probs.cpu().numpy()):
            id_to_prob[str(img_id)] = float(prob)
    return id_to_prob

print('Predict function ready')


Predict function ready


## Cell 12 — Threshold & Submission Helpers

In [12]:
def search_best_threshold(y_true, y_prob):
    best_thr, best_acc = 0.5, -1.0
    for thr in np.arange(0.25, 0.751, 0.005):
        acc = accuracy_score(y_true, (y_prob >= thr).astype(int))
        if acc > best_acc: best_acc, best_thr = acc, float(thr)
    return best_thr, best_acc

def compute_final_threshold(fold_thresholds, oof_probs, oof_labels):
    """ใช้ fold-average threshold ลด optimistic bias จาก OOF tuning"""
    avg_thr = float(np.mean(fold_thresholds))
    acc_050 = accuracy_score(oof_labels, (oof_probs >= 0.50).astype(int))
    acc_avg = accuracy_score(oof_labels, (oof_probs >= avg_thr).astype(int))
    print(f'  thr=0.500           → OOF acc={acc_050:.5f}')
    print(f'  thr={avg_thr:.3f} (fold avg) → OOF acc={acc_avg:.5f}')
    return avg_thr

def make_submission(sample_sub, id_to_prob, threshold, path):
    sub = sample_sub.copy()
    sub['id']   = sub['id'].astype(str)
    sub['prob'] = sub['id'].map(id_to_prob)
    missing = sub['prob'].isna().sum()
    if missing > 0: raise ValueError(f'Missing {missing} IDs!')
    sub['answer'] = (sub['prob'] >= threshold).astype(int)
    sub[['id','answer']].to_csv(path, index=False)
    print(f'Saved: {path} | dist={sub["answer"].value_counts().to_dict()}')

def make_loader(ds, batch_size, shuffle):
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=cfg.PREFETCH,
        worker_init_fn=worker_init_fn, drop_last=False,
    )

print('Helpers ready')


Helpers ready


## Cell 13 — โหลดข้อมูล & Build Cache

In [13]:
data_dir   = Path(cfg.DATA_DIR)
output_dir = Path(cfg.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

train_df   = pd.read_csv(data_dir / 'train.csv')
sample_sub = pd.read_csv(data_dir / 'sample_submission.csv')
sample_sub['id'] = sample_sub['id'].astype(str)

train_dir  = data_dir / 'train' / 'train'
test_dir   = data_dir / 'test'  / 'test'

test_df = pd.DataFrame({'image_name': sorted([p.name for p in test_dir.glob('*.jpg')])})
test_df['id'] = test_df['image_name'].str.replace('.jpg','',regex=False).astype(str)

missing = set(sample_sub['id']) - set(test_df['id'])
assert len(missing) == 0, f'ภาพหายไป {len(missing)} รูป'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Train: {len(train_df)} | Test: {len(test_df)} | Device: {device}')

train_cache = build_image_cache(train_dir, train_df['image_name'].tolist()) if cfg.CACHE_IMAGES else None
test_cache  = build_image_cache(test_dir,  test_df['image_name'].tolist())  if cfg.CACHE_IMAGES else None


Train: 2953 | Test: 1550 | Device: cuda
[Cache] โหลด 2953 ภาพ...


cache:   0%|          | 0/2953 [00:00<?, ?it/s]

[Cache] 4.84 GB used
[Cache] โหลด 1550 ภาพ...


cache:   0%|          | 0/1550 [00:00<?, ?it/s]

[Cache] 1.90 GB used


## Cell 14 — Training Loop (5-Fold CV)

In [ ]:
skf = StratifiedKFold(n_splits=cfg.N_SPLITS, shuffle=True, random_state=cfg.SEED)
oof_probs      = np.zeros(len(train_df), dtype=np.float32)
fold_id_probs  = []
fold_scores    = []
fold_thresholds= []

for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['class'])):
    print(f"\n{'='*55}\n  FOLD {fold}/{cfg.N_SPLITS-1}\n{'='*55}")

    fold_train = train_df.iloc[train_idx].reset_index(drop=True)
    fold_valid = train_df.iloc[valid_idx].reset_index(drop=True)

    train_ds = HouseDataset(fold_train, train_dir, build_train_transforms(cfg.IMAGE_SIZE), cache=train_cache)
    valid_ds = HouseDataset(fold_valid, train_dir, build_valid_transforms(cfg.IMAGE_SIZE), cache=train_cache)
    test_ds  = HouseDataset(test_df,   test_dir,  build_valid_transforms(cfg.IMAGE_SIZE), is_test=True, cache=test_cache)

    train_loader = make_loader(train_ds, cfg.TRAIN_BS, shuffle=True)
    valid_loader = make_loader(valid_ds, cfg.VALID_BS, shuffle=False)
    test_loader  = make_loader(test_ds,  cfg.VALID_BS, shuffle=False)

    model = HouseModel(cfg.MODEL_NAME, cfg.NUM_CLASSES, grad_ckpt=cfg.GRAD_CKPT).to(device)
    if cfg.CHANNELS_LAST: model = model.to(memory_format=torch.channels_last)

    optimizer = build_layerwise_optimizer(model, cfg.LR, cfg.WEIGHT_DECAY, cfg.LAYER_DECAY)
    eff_steps = math.ceil(len(train_loader) / cfg.GRAD_ACCUM)
    scheduler = build_scheduler(optimizer, eff_steps*cfg.EPOCHS, eff_steps*cfg.WARMUP_EPOCHS, cfg.MIN_LR, cfg.LR)
    scaler    = torch.amp.GradScaler(device.type)

    best_acc, best_probs, best_thr = -1.0, None, 0.5
    best_path  = output_dir / f'fold_{fold}_best.pt'
    no_improve = 0

    for epoch in range(cfg.EPOCHS):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device, scaler)
        vl_loss, vl_acc, vl_probs = validate(model, valid_loader, device)
        thr, thr_acc = search_best_threshold(fold_valid['class'].values, vl_probs)
        print(f'[fold {fold}] epoch {epoch+1}/{cfg.EPOCHS} '
              f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} '
              f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} '
              f'best_thr={thr:.3f}({thr_acc:.4f})')
        if vl_acc > best_acc:
            best_acc, best_probs, best_thr, no_improve = vl_acc, vl_probs.copy(), thr, 0
            torch.save({'state_dict': model.state_dict(), 'acc': vl_acc}, best_path)
        else:
            no_improve += 1
            if no_improve >= cfg.PATIENCE:
                print(f'[fold {fold}] Early stopping at epoch {epoch+1}'); break

    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)['state_dict'])
    id_to_prob = predict_loader(model, test_loader, device)

    oof_probs[valid_idx] = best_probs
    fold_id_probs.append(id_to_prob)
    fold_scores.append(best_acc)
    fold_thresholds.append(best_thr)
    print(f'[fold {fold}] best_acc={best_acc:.5f} | best_thr={best_thr:.3f}')

    train_loader._iterator = valid_loader._iterator = test_loader._iterator = None
    del train_loader, valid_loader, test_loader, model, optimizer, scheduler, scaler
    gc.collect(); torch.cuda.empty_cache()



  FOLD 0/4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

train:   0%|          | 0/148 [00:00<?, ?it/s]

valid:   0%|          | 0/19 [00:00<?, ?it/s]

...

predict:   0%|          | 0/49 [00:00<?, ?it/s]

[fold 3] best_acc=0.95763 | best_thr=0.450

  FOLD 4/4


train:   0%|          | 0/148 [00:00<?, ?it/s]

valid:   0%|          | 0/19 [00:00<?, ?it/s]

[fold 4] epoch 1/12 tr_loss=0.6364 tr_acc=0.6488 vl_loss=0.3589 vl_acc=0.9288 best_thr=0.505(0.9305)


## Cell 15 — Summary & สร้าง Submission

In [ ]:
oof_labels = train_df['class'].values
oof_acc    = accuracy_score(oof_labels, (oof_probs >= 0.5).astype(int))
print(f'OOF accuracy (thr=0.5) : {oof_acc:.5f}')
print(f'Fold scores            : {[round(x,5) for x in fold_scores]}')
print(f'Fold thresholds        : {[round(x,3) for x in fold_thresholds]}')

print('\nThreshold analysis:')
final_thr = compute_final_threshold(fold_thresholds, oof_probs, oof_labels)

# Ensemble
all_ids = list(fold_id_probs[0].keys())
mean_probs = {i: float(np.mean([fp[i] for fp in fold_id_probs])) for i in all_ids}

# OOF CSV
oof_df = train_df.copy()
oof_df['prob_1']    = oof_probs
oof_df['pred_050']  = (oof_probs >= 0.5).astype(int)
oof_df['pred_best'] = (oof_probs >= final_thr).astype(int)
oof_df.to_csv(output_dir / 'oof_predictions.csv', index=False)

make_submission(sample_sub, mean_probs, 0.5,       output_dir / 'submission_thr_050.csv')
make_submission(sample_sub, mean_probs, final_thr, output_dir / 'submission_best_thr.csv')


## Cell 16 — ดาวน์โหลด Submission

In [ ]:
from google.colab import files
files.download(str(output_dir / 'submission_thr_050.csv'))
files.download(str(output_dir / 'submission_best_thr.csv'))
